# Gold Layer Transformations

Notebook: nb_silver_to_gold_analytics

Purpose:
Transform curated Silver layer datasets into business-ready Gold layer analytics tables.

Source Tables:
- silver_orders_enriched

Target Tables:
- gold_customer_metrics

Architecture:
Bronze → Silver → Gold

###### <mark>**Read Silver Data**</mark>

In [2]:
# Read the enriched Silver dataset

silver_orders_df = spark.table("silver_orders_enriched")

# Inspect schema
silver_orders_df.printSchema()

# Preview data
silver_orders_df.show(3)

StatementMeta(, 1838c9b4-1f2c-4dee-9a9a-b3f7fdbfda69, 4, Finished, Available, Finished, False)

root
 |-- order_id: long (nullable = true)
 |-- product_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_value: double (nullable = true)
 |-- item_id: long (nullable = true)
 |-- quantity: long (nullable = true)
 |-- item_price: double (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- cost: double (nullable = true)
 |-- product_description: string (nullable = true)
 |-- payment_id: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- payment_amount: double (nullable = true)
 |-- line_revenue: double (nullable = true)
 |-- customer_lifetime_revenue: double (nullable = true)
 |-- customer_order_rank: integer (nullable = true)
 |-- running_customer_revenue: double (nullable = true)

+--------+----------+-----------+

###### **<mark>Customer Metrics Aggregation</mark>**

###### **<mark>Now creat the Gold aggregation</mark>**.

In [3]:
from pyspark.sql.functions import sum, count, avg

gold_customer_metrics_df = (
    silver_orders_df
        .groupBy("customer_id")
        .agg(
            count("order_id").alias("total_orders"),
            sum("line_revenue").alias("lifetime_revenue"),
            avg("line_revenue").alias("avg_order_value")
        )
)

StatementMeta(, 1838c9b4-1f2c-4dee-9a9a-b3f7fdbfda69, 5, Finished, Available, Finished, False)

###### <mark>**Validate Results**</mark>

In [4]:
gold_customer_metrics_df.show(10)

gold_customer_metrics_df.printSchema()

StatementMeta(, 1838c9b4-1f2c-4dee-9a9a-b3f7fdbfda69, 6, Finished, Available, Finished, False)

+-----------+------------+------------------+------------------+
|customer_id|total_orders|  lifetime_revenue|   avg_order_value|
+-----------+------------+------------------+------------------+
|         50|          36|30739.159999999996| 853.8655555555555|
|         94|          36|26007.860000000004| 722.4405555555556|
|        149|          15|          12964.77|           864.318|
|        421|          21|           16053.3| 764.4428571428571|
|       1038|          15|13170.229999999998| 878.0153333333332|
|       1174|           9|           3569.92|396.65777777777777|
|       1895|          21|          21082.06|1003.9076190476192|
|       2071|          30|23969.220000000005| 798.9740000000002|
|       2162|           6| 4585.429999999999| 764.2383333333332|
|       2477|          24|13612.979999999998| 567.2074999999999|
+-----------+------------+------------------+------------------+
only showing top 10 rows

root
 |-- customer_id: long (nullable = true)
 |-- total_orders:

###### **<mark>Write Gold Table</mark>**

In [5]:
gold_customer_metrics_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_customer_metrics")

StatementMeta(, 1838c9b4-1f2c-4dee-9a9a-b3f7fdbfda69, 7, Finished, Available, Finished, False)

###### **<mark>Verify Table</mark>**

In [6]:
spark.sql("""
SELECT COUNT(*)
FROM gold_customer_metrics
""").show()

spark.sql("""
SELECT *
FROM gold_customer_metrics
LIMIT 10
""").show()

StatementMeta(, 1838c9b4-1f2c-4dee-9a9a-b3f7fdbfda69, 8, Finished, Available, Finished, False)

+--------+
|count(1)|
+--------+
|  999646|
+--------+

+-----------+------------+------------------+------------------+
|customer_id|total_orders|  lifetime_revenue|   avg_order_value|
+-----------+------------+------------------+------------------+
|         61|          27|          17779.23|            658.49|
|        303|          15|15725.470000000001|1048.3646666666668|
|       1309|           9| 6459.980000000001| 717.7755555555557|
|       1812|           9|           3903.25|433.69444444444446|
|       1813|          18|           7214.05|400.78055555555557|
|       1968|          24|24004.060000000005|1000.1691666666669|
|       2312|          24|          14951.43|         622.97625|
|       2732|          21|15196.050000000005| 723.6214285714289|
|       2735|          33|           33093.9|1002.8454545454546|
|       2949|          18|13002.009999999998| 722.3338888888889|
+-----------+------------+------------------+------------------+



###### **<mark>Next Fact Table Build :- fact_sales</mark>**


In [7]:
fact_sales_df = spark.table("silver_orders_enriched") \
    .select(
        "order_id",
        "customer_id",
        "product_id",
        "order_date",
        "quantity",
        "line_revenue"
    )

StatementMeta(, 1838c9b4-1f2c-4dee-9a9a-b3f7fdbfda69, 9, Finished, Available, Finished, False)

###### **<mark>Validate Table</mark>** 

In [8]:
fact_sales_df.show(10)

StatementMeta(, 1838c9b4-1f2c-4dee-9a9a-b3f7fdbfda69, 10, Finished, Available, Finished, False)

+--------+-----------+----------+----------+--------+------------+
|order_id|customer_id|product_id|order_date|quantity|line_revenue|
+--------+-----------+----------+----------+--------+------------+
| 7649063|         11|     23435|2024-05-20|       5|     1122.35|
| 7649063|         11|     30497|2024-05-20|       5|      2207.5|
| 7649063|         11|     14227|2024-05-20|       4|      935.04|
| 7111529|         11|      3900|2024-09-16|       3|      384.72|
| 7111529|         11|     41529|2024-09-16|       1|      489.95|
| 7111529|         11|     34611|2024-09-16|       3|      859.02|
| 2751188|         11|     31490|2025-03-31|       5|      1980.0|
| 2751188|         11|      1905|2025-03-31|       3|      325.26|
| 2751188|         11|     40793|2025-03-31|       5|       202.0|
| 6812764|         11|     27821|2025-09-09|       3|     1356.96|
+--------+-----------+----------+----------+--------+------------+
only showing top 10 rows



###### **<mark>Save Fact Table</mark>**

In [9]:
fact_sales_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("fact_sales")

StatementMeta(, 1838c9b4-1f2c-4dee-9a9a-b3f7fdbfda69, 11, Finished, Available, Finished, False)

###### **<mark>Verify Table</mark>**

In [10]:
spark.sql("""
SELECT COUNT(*)
FROM fact_sales
""").show()

StatementMeta(, 1838c9b4-1f2c-4dee-9a9a-b3f7fdbfda69, 12, Finished, Available, Finished, False)

+--------+
|count(1)|
+--------+
|24000000|
+--------+



###### **<mark>Create Dimension Table: dim_customer</mark>**

In [11]:
dim_customer_df = (
    spark.table("silver_customers")
        .select(
            "customer_id",
            "customer_full_name",
            "email",
            "country",
            "customer_segment",
            "customer_signup_date",
            "loyalty_points"
        )
        .dropDuplicates(["customer_id"])
)

StatementMeta(, 1838c9b4-1f2c-4dee-9a9a-b3f7fdbfda69, 13, Finished, Available, Finished, False)

###### **<mark>Validate</mark>**

In [12]:
dim_customer_df.show(10)

StatementMeta(, 1838c9b4-1f2c-4dee-9a9a-b3f7fdbfda69, 14, Finished, Available, Finished, False)

+-----------+------------------+--------------------+-------+----------------+--------------------+--------------+
|customer_id|customer_full_name|               email|country|customer_segment|customer_signup_date|loyalty_points|
+-----------+------------------+--------------------+-------+----------------+--------------------+--------------+
|         12|  Christine Burton|normaadams@exampl...|     in|         regular|          2022-08-26|          5925|
|         26|    Anthony Taylor|pwilliams@example...|     in|             vip|          2021-08-22|         13264|
|         27|    Heather Martin|cgriffin@example.org|     in|         premium|          2022-07-29|          7354|
|         28|       Kathy Gomez|brandontrujillo@e...|     in|         regular|          2023-09-23|         16148|
|         31|       Jamie Silva|regina97@example.net|     in|             vip|          2025-09-01|          9629|
|         34|     Bruce Wallace|ericjohnson@examp...|     in|         regular|  

###### **<mark>Save Table</mark>**

In [13]:
dim_customer_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("dim_customer")

StatementMeta(, 1838c9b4-1f2c-4dee-9a9a-b3f7fdbfda69, 15, Finished, Available, Finished, False)

###### **<mark>Verify</mark>**

In [14]:
spark.sql("""
SELECT COUNT(*)
FROM dim_customer
""").show()

StatementMeta(, 1838c9b4-1f2c-4dee-9a9a-b3f7fdbfda69, 16, Finished, Available, Finished, False)

+--------+
|count(1)|
+--------+
| 1000000|
+--------+



###### **<mark>Next Step: Create dim_product</mark>**

###### <mark>**<mark>Products are another dimension table</mark>**.</mark>

In [15]:
dim_product_df = (
    spark.table("bronze_products")
        .select(
            "product_id",
            "product_name",
            "category",
            "price",
            "cost",
            "product_description"
        )
        .dropDuplicates(["product_id"])
)

StatementMeta(, 1838c9b4-1f2c-4dee-9a9a-b3f7fdbfda69, 17, Finished, Available, Finished, False)

###### **<mark>Validate</mark>**

In [16]:
dim_product_df.show(10)

StatementMeta(, 1838c9b4-1f2c-4dee-9a9a-b3f7fdbfda69, 18, Finished, Available, Finished, False)

+----------+------------+--------+------+------+--------------------+
|product_id|product_name|category| price|  cost| product_description|
+----------+------------+--------+------+------+--------------------+
|         7|      police|  Beauty| 83.77| 58.54|Whatever floor st...|
|        19|         and|  Sports|460.56|158.25|Program better in...|
|        22|      degree|   Books|124.62| 197.4|Leave dream keep ...|
|        26|         cup|Clothing| 31.77|101.04|Quickly task yard...|
|        29|         use|   Books|452.78|  9.11|Experience experi...|
|        31|      behind|   Books|336.88| 170.1|Administration se...|
|        32|       would|  Sports| 67.52|148.06|Cup attorney best...|
|        34|       class|   Books|239.31|173.17|Radio life time s...|
|        39|        safe|    Home|376.49| 13.27|Matter discussion...|
|        43|     however|  Beauty|415.56|179.49|Forward father pr...|
+----------+------------+--------+------+------+--------------------+
only showing top 10 

###### **<mark>Save as Delta Table</mark>**

In [17]:
dim_product_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("dim_product")

StatementMeta(, 1838c9b4-1f2c-4dee-9a9a-b3f7fdbfda69, 19, Finished, Available, Finished, False)

###### **<mark>Validate</mark>**

In [18]:
spark.sql("""
SELECT COUNT(*)
FROM dim_product
""").show()

StatementMeta(, 1838c9b4-1f2c-4dee-9a9a-b3f7fdbfda69, 20, Finished, Available, Finished, False)

+--------+
|count(1)|
+--------+
|   50000|
+--------+



###### **<mark>Final Dimension Table: dim_date</mark>**
###### **<mark>Create Date Range</mark>**

In [19]:
from pyspark.sql.functions import col, year, month, dayofmonth, quarter, date_format

date_df = spark.sql("""
SELECT explode(sequence(
    to_date('2024-01-01'),
    to_date('2026-12-31'),
    interval 1 day
)) AS date
""")

StatementMeta(, 1838c9b4-1f2c-4dee-9a9a-b3f7fdbfda69, 21, Finished, Available, Finished, False)

###### **<mark>Add Calendar Attributes</mark>**

In [20]:
dim_date_df = (
    date_df
    .withColumn("year", year(col("date")))
    .withColumn("month", month(col("date")))
    .withColumn("day", dayofmonth(col("date")))
    .withColumn("quarter", quarter(col("date")))
    .withColumn("weekday", date_format(col("date"), "E"))
)

StatementMeta(, 1838c9b4-1f2c-4dee-9a9a-b3f7fdbfda69, 22, Finished, Available, Finished, False)

###### **<mark>Validate</mark>**

In [21]:
dim_date_df.show(10)

StatementMeta(, 1838c9b4-1f2c-4dee-9a9a-b3f7fdbfda69, 23, Finished, Available, Finished, False)

+----------+----+-----+---+-------+-------+
|      date|year|month|day|quarter|weekday|
+----------+----+-----+---+-------+-------+
|2024-01-01|2024|    1|  1|      1|    Mon|
|2024-01-02|2024|    1|  2|      1|    Tue|
|2024-01-03|2024|    1|  3|      1|    Wed|
|2024-01-04|2024|    1|  4|      1|    Thu|
|2024-01-05|2024|    1|  5|      1|    Fri|
|2024-01-06|2024|    1|  6|      1|    Sat|
|2024-01-07|2024|    1|  7|      1|    Sun|
|2024-01-08|2024|    1|  8|      1|    Mon|
|2024-01-09|2024|    1|  9|      1|    Tue|
|2024-01-10|2024|    1| 10|      1|    Wed|
+----------+----+-----+---+-------+-------+
only showing top 10 rows



###### **<mark>Save Table</mark>**

In [22]:
dim_date_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("dim_date")

StatementMeta(, 1838c9b4-1f2c-4dee-9a9a-b3f7fdbfda69, 24, Finished, Available, Finished, False)

###### **<mark>Verify</mark>**

In [23]:
spark.sql("""
SELECT COUNT(*)
FROM dim_date
""").show()

StatementMeta(, 1838c9b4-1f2c-4dee-9a9a-b3f7fdbfda69, 25, Finished, Available, Finished, False)

+--------+
|count(1)|
+--------+
|    1096|
+--------+



In [1]:
spark.sql("SELECT COUNT(*) AS cnt, COUNT(DISTINCT customer_id) AS distinct_customers FROM fact_sales").show()
spark.sql("SELECT COUNT(*) AS cnt, COUNT(DISTINCT customer_id) AS distinct_customers FROM dim_customer").show()
spark.sql("SELECT COUNT(*) AS cnt, COUNT(DISTINCT product_id) AS distinct_products FROM fact_sales").show()
spark.sql("SELECT COUNT(*) AS cnt, COUNT(DISTINCT product_id) AS distinct_products FROM dim_product").show()
spark.sql("SELECT MIN(order_date), MAX(order_date), COUNT(DISTINCT order_date) FROM fact_sales").show()
spark.sql("SELECT MIN(date), MAX(date), COUNT(*) FROM dim_date").show()

StatementMeta(, ee289972-a026-4edf-a5cf-e3c89877d367, 3, Finished, Available, Finished, False)

+--------+------------------+
|     cnt|distinct_customers|
+--------+------------------+
|24000000|            999646|
+--------+------------------+

+-------+------------------+
|    cnt|distinct_customers|
+-------+------------------+
|1000000|           1000000|
+-------+------------------+

+--------+-----------------+
|     cnt|distinct_products|
+--------+-----------------+
|24000000|            50000|
+--------+-----------------+

+-----+-----------------+
|  cnt|distinct_products|
+-----+-----------------+
|50000|            50000|
+-----+-----------------+

+---------------+---------------+--------------------------+
|min(order_date)|max(order_date)|count(DISTINCT order_date)|
+---------------+---------------+--------------------------+
|     2024-03-07|     2026-03-07|                       731|
+---------------+---------------+--------------------------+

+----------+----------+--------+
| min(date)| max(date)|count(1)|
+----------+----------+--------+
|2024-01-01|2026-12-3

In [2]:
spark.sql("""
SELECT
  SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS null_customer_id,
  SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS null_product_id,
  SUM(CASE WHEN order_date IS NULL THEN 1 ELSE 0 END) AS null_order_date
FROM fact_sales
""").show()

StatementMeta(, ee289972-a026-4edf-a5cf-e3c89877d367, 4, Finished, Available, Finished, False)

+----------------+---------------+---------------+
|null_customer_id|null_product_id|null_order_date|
+----------------+---------------+---------------+
|               0|              0|              0|
+----------------+---------------+---------------+



###### **<mark>STEP 1: Build gold_product_category_sales**

**👉 This is your first real business KPI table</mark>**

**🧠 WHAT ARE WE BUILDING?**
Category-level insights:
- How much each category sells
- Which category performs best
- Revenue contribution

**🏗️ DATA FLOW**

fact_sales  → contains transactions
dim_product → contains category

JOIN → GROUP → AGGREGATE**

In [3]:
from pyspark.sql.functions import sum, count

gold_product_category_sales_df = (
    spark.table("fact_sales")
    .join(
        spark.table("dim_product"),
        "product_id"
    )
    .groupBy("category")
    .agg(
        count("order_id").alias("total_orders"),
        sum("quantity").alias("total_quantity"),
        sum("line_revenue").alias("total_revenue")
    )
)

StatementMeta(, ee289972-a026-4edf-a5cf-e3c89877d367, 5, Finished, Available, Finished, False)

In [4]:
gold_product_category_sales_df.orderBy("total_revenue", ascending=False).show(10)

StatementMeta(, ee289972-a026-4edf-a5cf-e3c89877d367, 6, Finished, Available, Finished, False)

+-----------+------------+--------------+--------------------+
|   category|total_orders|total_quantity|       total_revenue|
+-----------+------------+--------------+--------------------+
|     Beauty|     4043004|      12129497|3.0937313258999896E9|
|Electronics|     4015336|      12051114|3.0751650131500006E9|
|   Clothing|     4020416|      12056323|3.0747978014599934E9|
|      Books|     4001495|      12003681| 3.060233625599986E9|
|     Sports|     3993907|      11979799| 3.056601443679981E9|
|       Home|     3925842|      11777849| 3.003966442510019E9|
+-----------+------------+--------------+--------------------+



###### <mark>**Build a category-level aggregation table in the Gold layer**</mark>

In [5]:
gold_product_category_sales_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_product_category_sales")

StatementMeta(, ee289972-a026-4edf-a5cf-e3c89877d367, 7, Finished, Available, Finished, False)

In [6]:
spark.sql("""
SELECT *
FROM gold_product_category_sales
ORDER BY total_revenue DESC
LIMIT 10
""").show()

StatementMeta(, ee289972-a026-4edf-a5cf-e3c89877d367, 8, Finished, Available, Finished, False)

+-----------+------------+--------------+--------------------+
|   category|total_orders|total_quantity|       total_revenue|
+-----------+------------+--------------+--------------------+
|     Beauty|     4043004|      12129497|3.0937313258999896E9|
|Electronics|     4015336|      12051114|3.0751650131500006E9|
|   Clothing|     4020416|      12056323|3.0747978014599934E9|
|      Books|     4001495|      12003681| 3.060233625599986E9|
|     Sports|     3993907|      11979799| 3.056601443679981E9|
|       Home|     3925842|      11777849| 3.003966442510019E9|
+-----------+------------+--------------+--------------------+



<mark>**REAL-TIME USE CASE
E-commerce dashboard:**</mark>
- Top selling category
- Category revenue share
- Inventory planning

In [7]:
spark.sql("""
SELECT COUNT(*)
FROM gold_product_category_sales
WHERE category IS NULL
""").show()

StatementMeta(, ee289972-a026-4edf-a5cf-e3c89877d367, 9, Finished, Available, Finished, False)

+--------+
|count(1)|
+--------+
|       0|
+--------+



###### <mark>**👉 1. Daily Sales Trend (Time Intelligence)**</mark>
**gold_customer_profitability**

**🧠 WHAT WE ARE BUILDING**

<mark>For each customer, we’ll calculate:</mark>

- **- total orders**
- **- total quantity**
- **- total revenue**
- **- total estimated cost**
- **- total estimated profit**
- **- average profit per order**

**🏗️ DATA FLOW**
fact_sales + dim_product
→ bring cost from product dimension
→ derive row-level profit
→ aggregate by customer_id

**🧠 FORMULA**

<mark>At row level:</mark>

**estimated_cost = quantity * cost**
**estimated_profit = line_revenue - estimated_cost**

**Then aggregate at customer level.**

In [8]:
from pyspark.sql.functions import sum, countDistinct, round, col

gold_customer_profitability_df = (
    spark.table("fact_sales")
    .join(
        spark.table("dim_product").select("product_id", "cost"),
        "product_id"
    )
    .withColumn("estimated_cost", col("quantity") * col("cost"))
    .withColumn("estimated_profit", col("line_revenue") - col("estimated_cost"))
    .groupBy("customer_id")
    .agg(
        countDistinct("order_id").alias("total_orders"),
        sum("quantity").alias("total_quantity"),
        round(sum("line_revenue"), 2).alias("total_revenue"),
        round(sum("estimated_cost"), 2).alias("total_estimated_cost"),
        round(sum("estimated_profit"), 2).alias("total_estimated_profit"),
        round(sum("estimated_profit") / countDistinct("order_id"), 2).alias("avg_profit_per_order")
    )
)

StatementMeta(, ee289972-a026-4edf-a5cf-e3c89877d367, 10, Finished, Available, Finished, False)

In [9]:
gold_customer_profitability_df.orderBy(col("total_estimated_profit").desc()).show(10)

StatementMeta(, ee289972-a026-4edf-a5cf-e3c89877d367, 11, Finished, Available, Finished, False)

+-----------+------------+--------------+-------------+--------------------+----------------------+--------------------+
|customer_id|total_orders|total_quantity|total_revenue|total_estimated_cost|total_estimated_profit|avg_profit_per_order|
+-----------+------------+--------------+-------------+--------------------+----------------------+--------------------+
|     169346|          21|           195|     62692.38|            19248.47|              43443.91|             2068.76|
|     666937|          21|           200|     58984.48|            16553.79|              42430.69|             2020.51|
|      26881|          23|           204|     59417.22|            19075.79|              40341.43|             1753.98|
|     843120|          24|           211|     57572.84|            17610.61|              39962.23|             1665.09|
|     331870|          22|           225|     57752.61|            17954.25|              39798.36|             1809.02|
|     577978|          18|      

In [10]:
gold_customer_profitability_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_customer_profitability")

StatementMeta(, ee289972-a026-4edf-a5cf-e3c89877d367, 12, Finished, Available, Finished, False)

In [11]:
spark.sql("""
SELECT *
FROM gold_customer_profitability
ORDER BY total_estimated_profit DESC
LIMIT 10
""").show()

StatementMeta(, ee289972-a026-4edf-a5cf-e3c89877d367, 13, Finished, Available, Finished, False)

+-----------+------------+--------------+-------------+--------------------+----------------------+--------------------+
|customer_id|total_orders|total_quantity|total_revenue|total_estimated_cost|total_estimated_profit|avg_profit_per_order|
+-----------+------------+--------------+-------------+--------------------+----------------------+--------------------+
|     169346|          21|           195|     62692.38|            19248.47|              43443.91|             2068.76|
|     666937|          21|           200|     58984.48|            16553.79|              42430.69|             2020.51|
|      26881|          23|           204|     59417.22|            19075.79|              40341.43|             1753.98|
|     843120|          24|           211|     57572.84|            17610.61|              39962.23|             1665.09|
|     331870|          22|           225|     57752.61|            17954.25|              39798.36|             1809.02|
|     577978|          18|      

Data Quality Check 

In [12]:
spark.sql("""
SELECT COUNT(*)
FROM gold_customer_profitability
WHERE total_estimated_profit IS NULL
""").show()

StatementMeta(, ee289972-a026-4edf-a5cf-e3c89877d367, 14, Finished, Available, Finished, False)

+--------+
|count(1)|
+--------+
|       0|
+--------+



**🔥 REAL-WORLD USE CASE**

<mark>**Used for:**</mark>

- VIP customer targeting
- retention strategy
- discount planning
- margin analysis

*******************************************************************************************

###### <mark>**let’s build the time-based Gold mart:**</mark>

**🚀 NEXT: <mark>gold_daily_sales</mark>**

This table is very important because almost every dashboard needs:

- daily revenue
- daily orders
- daily quantity
- daily active customers

**🧠 WHAT WE ARE BUILDING**

**One row per order_date with:**

- total orders
- total quantity
- total revenue
- active customers

**This gives you trend analysis and time-series reporting.**

In [13]:
from pyspark.sql.functions import countDistinct, sum, round

gold_daily_sales_df = (
    spark.table("fact_sales")
    .groupBy("order_date")
    .agg(
        countDistinct("order_id").alias("total_orders"),
        sum("quantity").alias("total_quantity"),
        round(sum("line_revenue"), 2).alias("total_revenue"),
        countDistinct("customer_id").alias("active_customers")
    )
)

StatementMeta(, ee289972-a026-4edf-a5cf-e3c89877d367, 15, Finished, Available, Finished, False)

<mark>**Validate Output**</mark>

In [14]:
gold_daily_sales_df.orderBy("order_date").show(10)

StatementMeta(, ee289972-a026-4edf-a5cf-e3c89877d367, 16, Finished, Available, Finished, False)

+----------+------------+--------------+-------------+----------------+
|order_date|total_orders|total_quantity|total_revenue|active_customers|
+----------+------------+--------------+-------------+----------------+
|2024-03-07|       10922|         98729|2.525442765E7|           10873|
|2024-03-08|       10919|         98016|2.506046778E7|           10865|
|2024-03-09|       10956|         98554|2.525037195E7|           10898|
|2024-03-10|       11174|        100671| 2.56929193E7|           11115|
|2024-03-11|       11148|        100543|2.565544229E7|           11072|
|2024-03-12|       10916|         98280|2.508907128E7|           10849|
|2024-03-13|       11026|         99384|2.529892233E7|           10978|
|2024-03-14|       10982|         98890|2.524748658E7|           10908|
|2024-03-15|       10975|         99114|2.539333796E7|           10931|
|2024-03-16|       10802|         97225|2.483289788E7|           10760|
+----------+------------+--------------+-------------+----------

In [15]:
gold_daily_sales_df.orderBy(gold_daily_sales_df["order_date"].desc()).show(10)

StatementMeta(, ee289972-a026-4edf-a5cf-e3c89877d367, 17, Finished, Available, Finished, False)

+----------+------------+--------------+-------------+----------------+
|order_date|total_orders|total_quantity|total_revenue|active_customers|
+----------+------------+--------------+-------------+----------------+
|2026-03-07|       10916|         98164| 2.49694261E7|           10864|
|2026-03-06|       10888|         98099|2.510248487E7|           10819|
|2026-03-05|       10747|         96700|2.465384091E7|           10692|
|2026-03-04|       10868|         97817|2.506251205E7|           10813|
|2026-03-03|       10887|         97694|2.478967416E7|           10826|
|2026-03-02|       10868|         97756|2.496988713E7|           10809|
|2026-03-01|       10940|         98159|2.500268539E7|           10874|
|2026-02-28|       10861|         97404|2.487917181E7|           10789|
|2026-02-27|       11072|         99656|2.542507686E7|           11012|
|2026-02-26|       10958|         98317|2.515155387E7|           10900|
+----------+------------+--------------+-------------+----------

In [16]:
gold_daily_sales_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_daily_sales")

StatementMeta(, ee289972-a026-4edf-a5cf-e3c89877d367, 18, Finished, Available, Finished, False)

In [17]:
spark.sql("""
SELECT *
FROM gold_daily_sales
ORDER BY order_date
LIMIT 10
""").show()

StatementMeta(, ee289972-a026-4edf-a5cf-e3c89877d367, 19, Finished, Available, Finished, False)

+----------+------------+--------------+-------------+----------------+
|order_date|total_orders|total_quantity|total_revenue|active_customers|
+----------+------------+--------------+-------------+----------------+
|2024-03-07|       10922|         98729|2.525442765E7|           10873|
|2024-03-08|       10919|         98016|2.506046778E7|           10865|
|2024-03-09|       10956|         98554|2.525037195E7|           10898|
|2024-03-10|       11174|        100671| 2.56929193E7|           11115|
|2024-03-11|       11148|        100543|2.565544229E7|           11072|
|2024-03-12|       10916|         98280|2.508907128E7|           10849|
|2024-03-13|       11026|         99384|2.529892233E7|           10978|
|2024-03-14|       10982|         98890|2.524748658E7|           10908|
|2024-03-15|       10975|         99114|2.539333796E7|           10931|
|2024-03-16|       10802|         97225|2.483289788E7|           10760|
+----------+------------+--------------+-------------+----------

In [18]:
spark.sql("""
SELECT
    COUNT(*) AS total_days,
    MIN(order_date) AS min_date,
    MAX(order_date) AS max_date
FROM gold_daily_sales
""").show()

StatementMeta(, ee289972-a026-4edf-a5cf-e3c89877d367, 20, Finished, Available, Finished, False)

+----------+----------+----------+
|total_days|  min_date|  max_date|
+----------+----------+----------+
|       731|2024-03-07|2026-03-07|
+----------+----------+----------+



<mark>**🧠 WHY THIS IS IMPORTANT**</mark>

**This supports:**

- trend charts
- seasonality checks
- campaign impact analysis
- daily monitoring dashboards

********************************************************************

###### <mark>**👉 gold_top_customers**</mark>

**<mark>This is very common in real-world dashboards.</mark>**

In [19]:
from pyspark.sql.functions import col

gold_top_customers_df = (
    spark.table("gold_customer_profitability")
    .orderBy(col("total_estimated_profit").desc())
    .limit(100)
)

StatementMeta(, ee289972-a026-4edf-a5cf-e3c89877d367, 21, Finished, Available, Finished, False)

In [20]:
gold_top_customers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_top_customers")

StatementMeta(, ee289972-a026-4edf-a5cf-e3c89877d367, 22, Finished, Available, Finished, False)

<mark>**Why this matters:**</mark>

<mark>**Business:**</mark>
- Identify VIP customers
- Loyalty programs
- Marketing targeting